## Part 2: Practical Challenge

**Scenario: "The Q3 Review Pack"**

Your Lesson 1.9 analysis landed well. The owner has now asked for a short pack ahead of the Q3 review, with four specific questions. Same data, new questions.

Start a new notebook in notebooks/ and set up:

In [5]:
import pandas as pd
import numpy as np

sales = pd.read_csv("../data/daily_sales.csv", parse_dates=["date"])
outlets = pd.read_csv("../data/outlets.csv", parse_dates=["opened_date"])
roster = pd.read_csv("../data/roster.csv", parse_dates=["week_start"])
targets_wide = pd.read_csv("../data/targets_wide.csv")


display(sales.head())
display(outlets)
display(roster.head())
display(targets_wide)


,date,outlet_id,daypart,tickets,items,revenue_sgd
0,2024-01-01,OUT-01,Morning,122,168,762.73
1,2024-01-01,OUT-01,Midday,53,76,537.89
2,2024-01-01,OUT-01,Evening,17,27,164.35
3,2024-01-02,OUT-01,Morning,187,279,1184.16
4,2024-01-02,OUT-01,Midday,68,96,692.68


,outlet_id,outlet_name,region,opened_date,seats,monthly_rent_sgd
0,OUT-01,Raffles Place,Central,2019-04-15,28,8200
1,OUT-02,Tampines Mall,East,2020-09-01,42,6800
2,OUT-03,Marina Bay,Central,2021-02-10,24,9600
3,OUT-04,Holland Village,Central,2023-06-20,36,6200
4,OUT-06,Sentosa Cove,South,2025-08-01,30,7500


,outlet_id,week_start,staff_hours,headcount
0,OUT-01,2024-01-01,473.0,14
1,OUT-01,2024-01-08,421.6,12
2,OUT-01,2024-01-15,459.7,14
3,OUT-01,2024-01-22,446.1,13
4,OUT-01,2024-01-29,441.9,13


,outlet_id,2024-01,2024-02,2024-03,2024-04,2024-05,2024-06,2024-07,2024-08,2024-09,2024-10,2024-11,2024-12,2025-01,2025-02,2025-03,2025-04,2025-05,2025-06
0,OUT-01,51800.0,52000.0,52100.0,52300.0,52500.0,52700.0,52800.0,53000.0,53200.0,53300.0,53500.0,53700.0,53900.0,54000.0,54200.0,54400.0,54600.0,54800.0
1,OUT-02,45900.0,46000.0,46200.0,46300.0,46500.0,46600.0,46800.0,47000.0,47100.0,47300.0,47400.0,47600.0,47700.0,47900.0,48000.0,48200.0,48400.0,48500.0
2,OUT-03,46000.0,46100.0,46300.0,46400.0,46600.0,46700.0,46900.0,47000.0,47200.0,47300.0,47500.0,47600.0,47800.0,47900.0,48100.0,48300.0,48400.0,48600.0
3,OUT-04,32000.0,32100.0,32200.0,32300.0,32400.0,32500.0,32700.0,32800.0,32900.0,33000.0,33100.0,33200.0,33300.0,33400.0,33500.0,33600.0,33700.0,33800.0


**Challenge 1: "Is Tampines Mall as steady as it looks?" (Time)**

Tampines Mall (`OUT-02`) has been flat all year, which everyone has read as "stable".

Tasks:

 1. Build a weekly revenue series for `OUT-02` (weeks starting Monday).
 2. Add a 4-week rolling mean column beside it.
 3. Report its best and worst weeks by revenue.
 4. In one sentence: is "stable" the right word, or is it "noisy around a flat average"? Which column did you use to decide, and why?

In [33]:
# outlet 2 weekly revenue start on Monday
outlet2 = sales[sales["outlet_id"] == "OUT-02"]

weekly_rev_mon = outlet2.groupby(pd.Grouper(key="date", freq="W-MON", label="left"))["revenue_sgd"].sum()

# Drop the two partial weeks (not full 7days week)
weekly_rev_mon = weekly_rev_mon.iloc[1:-1]

display(weekly_rev_mon)

rolling_mean_4wk = pd.DataFrame({
    "revenue": weekly_rev_mon.round(0),
    "rolling_4w": weekly_rev_mon.rolling(4).mean().round(0),
})

display(rolling_mean_4wk.tail(8))
print("\nbest week: ", weekly_rev_mon.idxmax().date(), f"${weekly_rev_mon.max():,.0f}")
print("worst week:", weekly_rev_mon.idxmin().date(), f"${weekly_rev_mon.min():,.0f}")
print(f"\nspread: best is {weekly_rev_mon.max() / weekly_rev_mon.min() - 1:.0%} above worst")

date
2024-01-01    10807.66
2024-01-08    10661.09
2024-01-15    10910.79
2024-01-22    10735.09
2024-01-29    10913.29
                ...   
2025-05-19    11565.57
2025-05-26    11103.59
2025-06-02    10280.85
2025-06-09    10522.51
2025-06-16    10698.19
Freq: W-MON, Name: revenue_sgd, Length: 77, dtype: float64

,revenue,rolling_4w
date,,
2025-04-28,10435.0,10930.0
2025-05-05,11205.0,10905.0
2025-05-12,11599.0,11114.0
2025-05-19,11566.0,11201.0
2025-05-26,11104.0,11368.0
2025-06-02,10281.0,11137.0
2025-06-09,10523.0,10868.0
2025-06-16,10698.0,10651.0



best week:  2024-08-12 $11,899
worst week: 2024-04-29 $9,625

spread: best is 24% above worst


**Challenge 2: "Are we paying for staff we do not need?" (Joins)**

Tasks:

 1. Build weekly revenue per outlet (one row per outlet per week, weeks starting Monday).
 2. Merge it with `roster` on both keys, and prove the merge did not lose or duplicate rows.
 3. Add a `rev_per_staff_hour` column.
 4. Find the 10 worst weeks by `rev_per_staff_hour` across the whole chain. Which outlet dominates that list, and when do those weeks cluster?

In [40]:
weekly_sales = (
    sales.groupby(["outlet_id", pd.Grouper(key="date", freq="W-MON", label="left")])["revenue_sgd"]
    .sum()
    .reset_index()
    .rename(columns={"date": "week_start"})
)

staffed = weekly_sales.merge(roster, on=["outlet_id", "week_start"], how="inner", validate="one_to_one")

print(f"sales weeks: {len(weekly_sales)}  roster weeks: {len(roster)}  matched: {len(staffed)}")

staffed["rev_per_staff_hour"] = (staffed["revenue_sgd"] / staffed["staff_hours"]).round(2)

worst = staffed.nsmallest(10, "rev_per_staff_hour")

print(worst[["outlet_id", "week_start", "revenue_sgd", "staff_hours", "rev_per_staff_hour"]])
print("\noutlet counts in the worst 10:")
print(worst["outlet_id"].value_counts())

sales weeks: 330  roster weeks: 325  matched: 325
    outlet_id week_start  revenue_sgd  staff_hours  rev_per_staff_hour
232    OUT-03 2025-06-16      7897.27        455.9               17.32
203    OUT-03 2024-11-25      7664.01        437.1               17.53
324    OUT-05 2025-05-26      3248.18        182.9               17.76
230    OUT-03 2025-06-02      7257.87        407.3               17.82
212    OUT-03 2025-01-27      7771.50        429.9               18.08
229    OUT-03 2025-05-26      7203.80        389.2               18.51
206    OUT-03 2024-12-16      7714.06        409.6               18.83
223    OUT-03 2025-04-14      7644.26        403.9               18.93
233    OUT-03 2025-06-23      7264.20        378.9               19.17
205    OUT-03 2024-12-09      7743.65        399.6               19.38

outlet counts in the worst 10:
OUT-03    9
OUT-05    1
Name: outlet_id, dtype: int64


**Challenge 3: "Who is actually hitting target?" (Reshape + aggregate)**

Tasks:

 1. `melt` `targets_wide` into long format.
 2. Build monthly actual revenue per outlet, and join the targets on.
 3. Add a boolean `hit_target` column.
 4. Produce a table with one row per outlet showing months hit, months missed, and hit rate as a percentage. Sort it worst-first.
 5. One sentence: does the hit rate tell the same story as the revenue trend from class? If not, why not?

In [48]:
targets = targets_wide.melt(id_vars="outlet_id", var_name="month", value_name="target_sgd")
display(targets.head())

actual = (
    sales.groupby(["outlet_id", sales["date"].dt.to_period("M").astype(str)])["revenue_sgd"]
    .sum()
    .reset_index()
)
actual.columns = ["outlet_id", "month", "revenue_sgd"]
display(actual.head())

perf = actual.merge(targets, on=["outlet_id", "month"], how="left")
perf["hit_target"] = perf["revenue_sgd"] >= perf["target_sgd"]


# .dropna(subset=["target_sgd"]) -> ensures only analyze the months with a target
# ensure only drop when target_sgd = NaN, if there is NaN in others just leave it
result = perf.dropna(subset=["target_sgd"]).groupby("outlet_id").agg(
    months=("hit_target", "size"),
    hit=("hit_target", "sum"),
)
result["missed"] = result["months"] - result["hit"]
result["hit_rate_pct"] = (result["hit"] / result["months"] * 100).round(1)

display(result.sort_values("hit_rate_pct"))

,outlet_id,month,target_sgd
0,OUT-01,2024-01,51800.0
1,OUT-02,2024-01,45900.0
2,OUT-03,2024-01,46000.0
3,OUT-04,2024-01,32000.0
4,OUT-01,2024-02,52000.0


,outlet_id,month,revenue_sgd
0,OUT-01,2024-01,53744.04
1,OUT-01,2024-02,49725.46
2,OUT-01,2024-03,51932.42
3,OUT-01,2024-04,53412.10
4,OUT-01,2024-05,54228.57


,months,hit,missed,hit_rate_pct
outlet_id,,,,
OUT-01,18,5,13,27.8
OUT-03,18,5,13,27.8
OUT-02,18,7,11,38.9
OUT-04,18,16,2,88.9


**Challenge 4: "What exactly did Marina Bay lose?" (Aggregate)**

Revenue fell 34%. But which trade fell — the morning commuters, the lunch crowd, or the evening?

Tasks:

 1. Filter to `OUT-03` and split the data into two windows: 2024 H1 and 2025 H1.
 2. For each window, build the revenue mix by `daypart` as a percentage of that window's total.
 3. Put the two side by side and compute the change in percentage points.
 4. Then do it again in dollars per day rather than percentages.
 5. One sentence: which of the two tables would you put in the pack, and why? (They do not tell the same story.)



In [68]:
marina = sales[sales["outlet_id"] == "OUT-03"]

h1_2024 = marina[(marina["date"] >= "2024-01-01") & (marina["date"] <= "2024-06-30")]
h1_2025 = marina[(marina["date"] >= "2025-01-01") & (marina["date"] <= "2025-06-30")]

# Calculate revenue share/mix % by daypart (shift) for each half year (revenue mix)
# df is a temperary name which can be recycle later, replace with actual df name -> df(h1_2024), df(h1_2025) etc
def rev_mix_pct(df):
    daypart_rev = df.groupby("daypart")["revenue_sgd"].sum()
    return (daypart_rev / daypart_rev.sum() * 100).round(1)

# Calculate average revenue per day for each daypart (shift)
def rev_per_day(df):
    daypart_rev = df.groupby("daypart")["revenue_sgd"].sum()
    return (daypart_rev / df["date"].nunique()).round(0)

order = ["Morning", "Midday", "Evening"]  # daypart shift

rev_share = pd.DataFrame({"2024_h1_pct": rev_mix_pct(h1_2024), "2025_h1_pct": rev_mix_pct(h1_2025)}).loc[order]
rev_share["yoy"] = (rev_share["2025_h1_pct"] - rev_share["2024_h1_pct"]).round(1)
print(rev_share)

revenue = pd.DataFrame({"2024_h1_per_day": rev_per_day(h1_2024), "2025_h1_per_day": rev_per_day(h1_2025)}).loc[order]
revenue["yoy_pct"] = ((revenue["2025_h1_per_day"] / revenue["2024_h1_per_day"] - 1) * 100).round(1)
print("\n", revenue)

         2024_h1_pct  2025_h1_pct  yoy
daypart                               
Morning         48.4         47.6 -0.8
Midday          37.8         38.5  0.7
Evening         13.8         13.9  0.1

          2024_h1_per_day  2025_h1_per_day  yoy_pct
daypart                                           
Morning            749.0            534.0    -28.7
Midday             585.0            432.0    -26.2
Evening            214.0            156.0    -27.1


🏆 **Stretch Challenge (optional)**

The competitor opened on 4 November 2024. Did it hurt every day of the week equally, or is the damage concentrated?

Build a table with `weekday` down the side and two columns — average daily revenue for the 8 weeks before and the 8 weeks after — plus the percentage change. Then say what you would do with the answer.


In [69]:
marina_compare = sales[sales["outlet_id"] == "OUT-03"].copy()
marina_compare["weekday"] = marina_compare["date"].dt.day_name()

# One row per day first, or you will be averaging dayparts instead of days.
daily = marina_compare.groupby(["date", "weekday"])["revenue_sgd"].sum().reset_index()

before = daily[(daily["date"] >= "2024-09-09") & (daily["date"] < "2024-11-04")]
after = daily[(daily["date"] >= "2024-11-04") & (daily["date"] < "2024-12-30")]

weekday_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]

compare = pd.DataFrame({
    "before": before.groupby("weekday")["revenue_sgd"].mean().round(0),
    "after": after.groupby("weekday")["revenue_sgd"].mean().round(0),
}).loc[weekday_order]
compare["change_pct"] = ((compare["after"] / compare["before"] - 1) * 100).round(1)

print(compare)

           before   after  change_pct
weekday                              
Monday     1685.0  1382.0       -18.0
Tuesday    1864.0  1389.0       -25.5
Wednesday  1754.0  1306.0       -25.5
Thursday   1744.0  1410.0       -19.2
Friday     1642.0  1261.0       -23.2
Saturday    863.0   709.0       -17.8
Sunday      638.0   505.0       -20.8
